# Config

In [ ]:
'''#MOana specific
import os, subprocess
print("PID:", os.getpid())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("hostname:", subprocess.run(["hostname"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
'''

In [ ]:
conda run -n mega_sam python plot_vggt_extrinsic.py \
  "Data/01_walk/034_VGGT_output/01_walk_sparse_50_02_full_02/extrinsic.npy" \
  vggt_extrinsic_chart.png


In [ ]:
'''
#MOANA ONLY
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
#check for errors
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
'''

In [ ]:
# Cell 0 — CONFIG

#case
case_name = "304_met_multi_4"
experiment = "pipetest_01"
asset_name = f"{case_name}_{experiment}"
import json
from pathlib import Path
import numpy as np
%load_ext autoreload
%autoreload 2
import sys, os
from dotenv import load_dotenv
project_root = Path.cwd()
while not (project_root / "src" / "A_Config.py").exists():
    if project_root == project_root.parent:
        raise RuntimeError("couldn't find repo root (src/A_Config.py) above cwd")
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root / "src" / "Experiments" / "templates"))
from A_Config import (
    set_case, case_dir, source_dir, query_dir, sam3_masks_dir,
    frames_for_cam_poses_dir, frames_for_recon_dir, vggt_o_output_dir,
    predictions_path, ply_dir, reg_dir, csv_path_gps, csv_path_pose,
    assets_dir,
)

print("project_root:", project_root)
print(project_root, project_root.exists())
load_dotenv(project_root / ".env")

set_case(case_name, experiment)

case_dir_ = case_dir()
source_dir_ = source_dir()
query_dir_ = query_dir()
masks_dir = sam3_masks_dir()
frames_for_cam_poses = frames_for_cam_poses_dir()
frames_for_recon = frames_for_recon_dir()
VGGT_O_output_dir = vggt_o_output_dir()
ply_dir_ = ply_dir()
reg_dir_ = reg_dir()
csv_path_GPS = csv_path_gps()
csv_path_pose_ = csv_path_pose()

FRAME_NAME_FMT = "{:04d}.jpg"
# video loading
from Two2D.B_video_processing import video_fps
# Auto-find the video in 010_source
videos = list(source_dir_.glob("*.mp4"))
print(source_dir_)
assert len(videos) == 1, f"Expected 1 video, found {len(videos)}: {videos}"
video_path = videos[0]
print(f"Case    : {case_name}")
print(f"Video   : {video_path.name}")
fps = video_fps(video_path)
print(f"video fps: {fps:.3f}")
import cv2; total_frames = int(cv2.VideoCapture(str(video_path)).get(cv2.CAP_PROP_FRAME_COUNT))
print("total_frames",total_frames )
#reporting
from C_CSV_report import add_to_report
overview = {
    "Video_name"      : video_path.name, 
    "Title"      : f"{case_name}_{experiment}"
}
add_to_report(overview)
api_key = os.getenv("STREETVIEW_API_KEY")


In [ ]:
#FLAGS dict and 'producer dict
flags = {
    "VID_TO_TEXT": False,
    "TRACKING": False,
    "FRAME_SELECTION": True,
    "RECON_CAM_POSES": False,
    "RECON_3D": False,
    "GOOGLE_MAP": False,
    "BEV_MAP" : True,
    "PROJECTION_MAP": False,
    "MAKE_MOVIE": False,
    "MAKE_MOVIE_CUTS" : False,
    "MASK_OVERLAYS": False ,
    "PHYSICS_OVERLAYS" : False,
    "MULTICAM" : False
}
MENU_KEYS = {"TRACKING","RECON_CAM_POSES", "RECON_3D",
                          "GOOGLE_MAP", "BEV_MAP" , "PROJECTION_MAP", "MASK_OVERLAYS", "MULTICAM"}
producer_flags = {k: v for k, v in flags.items() if k in MENU_KEYS}

# AI First Pass
## Gem, vid to text

In [ ]:
#turn a video into a text description
if flags["VID_TO_TEXT"] == True:
    from Two2D.A_gemini_v02 import gemini_vid_to_text
    gem_summaries = gemini_vid_to_text(video_path, case_name, query_dir())
else:
    print("Cell disabled")



In [ ]:
#2d ANALYSIS - JUST GPS now!
from D_2d_analysis import analysis_2D
analysis_2d_for_decisions = analysis_2D(video_path,  api_key)
print(analysis_2d_for_decisions)

## Producer 1

In [ ]:
Question = "what was the main event in this video, give a detailed summary?"

In [ ]:
#PRODUCER DRAFT 1
sys.path.insert(0, str(project_root / "src" / "claude_agent"))
from claude_agent.producer_agent_execution import run_producer_agent, build_producer_prompt_draft1
prompt = build_producer_prompt_draft1(Question, analysis_2d_for_decisions, producer_flags)
paper_edit_json_path = run_producer_agent(prompt, mode ="draft")


In [ ]:
# DRAFT TIDY UP
# Re-run this cell on its own to regenerate the HTML/flags after editing and correct mistakes
import importlib
import json
sys.path.insert(0, str(project_root / "src" / "claude_agent"))
import render_paper_edit
importlib.reload(render_paper_edit)
from render_paper_edit import render_and_save, derive_and_save_flags, paper_edit_path
try:
    paper_edit_json_path  # from run_producer_agent, if the kernel still has it
except NameError:
    paper_edit_json_path = paper_edit_path()

html_path = render_and_save(paper_edit_json_path)
print(f"\n=== Rendered HTML ===\n  {html_path}")

flags_path = derive_and_save_flags(
    paper_edit_json_path, producer_flags.keys(), fps=fps,
    gps_signal=analysis_2d_for_decisions.get("GPS_signal"),
)
producer_output_flags = json.loads(flags_path.read_text(encoding="utf-8"))
flags.update(producer_output_flags)  # Producer owns MENU_KEYS; the rest stay operator-set
print(f"\n=== Derived flags (from {flags_path.name}) ===")
print(json.dumps(producer_output_flags, indent=2))


# Tracking

In [ ]:
#ROBOFLOW TRACKING SAM3 PIPELINE

if flags["TRACKING"]:
    sys.path.insert(0, str(project_root / "src" / "claude_agent"))
    from run_models.A_ROBOFLOW_SAM3 import track_subject_sam3
  
    analysis_2d_for_decisions.update (track_subject_sam3(video_path)) # add to the dict
    tracker = "SAM3"
    print(analysis_2d_for_decisions)
else:
    print("Cell disabled")


In [ ]:
#ROBOFLOW TRACKING SAM3 MANUAL controls
manual_sam = False
if manual_sam == True:
    sys.path.insert(0, str(project_root / "src" / "claude_agent"))
    from run_models.A_ROBOFLOW_SAM3 import track_subject_sam3, run_sam3_manual
    analysis_2d_for_decisions.update (run_sam3_manual(video_path, "man")) # add to the dict
    tracker = "SAM3"
    print(analysis_2d_for_decisions)


In [ ]:
print(analysis_2d_for_decisions)

In [ ]:
#mask checker (overlays)
mask_check = True
if mask_check ==True:
    
    print(masks_dir)
    from Two2D.W_video_editor import video_overlay_edit
    for i, mask in enumerate(p for p in masks_dir.iterdir() if p.is_dir()):
        video = video_overlay_edit(video_path, 0, 609, mask, 0,609, name = f"overlay_tst_{i}")
    

# RECON
## Frame Selection

In [ ]:
#FRAME SELECTION POSES - Frame selection to make the cam poses
if flags["RECON_CAM_POSES"] == True and flags["FRAME_SELECTION"] == True:
    from render_paper_edit import recon_window_frames
    start_cam_recon , end_cam_recon = recon_window_frames(paper_edit_json_path=None, fps=None, flag="RECON_CAM_POSES")
    capacity = 200
    recon_dur = end_cam_recon - start_cam_recon
    cam_recon_interframe_interval =   (recon_dur+capacity-1) // capacity
    print("interval" , cam_recon_interframe_interval)

    # Parameters
    fps = fps
    interval     = cam_recon_interframe_interval/fps      # seconds between samples - expressed as frames to skip, per second(frame rate)
    window       = fps/2     # search window for sharpness
    min_sharp    = 0.0    # raise this after inspecting selection_log.json


    import sys
    sys.path.insert(0, str(project_root / "src"))

    from Two2D.B_video_processing import image_sequencer

    results = image_sequencer(
        #video_path    = str(video_path),
        video_path = str(video_path),
        output_dir    = str(frames_for_cam_poses),
        #res           = res,
        #interval_sec  = interval,
        search_window = window,
        min_sharpness = min_sharp,
        start_frame  = 0,
        #end_frame    = end_cam_recon
    )

    print(f"\n{len(results)} frames written to {frames_for_recon}")
else:
    print("Cell disabled")

In [ ]:
#FRAMES FOR RECON SELECTION - Frame selection to make the recon
if flags["RECON_3D"] == True and flags["FRAME_SELECTION"] == True:
    from render_paper_edit import recon_window_frames
    start_3D_recon , end_3D_recon = recon_window_frames(paper_edit_json_path=None, fps=None, flag="RECON_3D")
    capacity = 200
    recon_dur = end_3D_recon - start_3D_recon
    cam_recon_interframe_interval =   (recon_dur+capacity-1) // capacity
    print("interval" , cam_recon_interframe_interval)

    # Parameters
    fps = fps
    interval     = cam_recon_interframe_interval/fps      # seconds between samples - expressed as frames to skip, per second(frame rate)
    window       = fps/2     # search window for sharpness
    min_sharp    = 0.0    # raise this after inspecting selection_log.json
   
    import sys
    sys.path.insert(0, str(project_root / "src"))

    from Two2D.B_video_processing import image_sequencer

    results = image_sequencer(
        #video_path    = str(video_path),
        video_path = str(video_path),
        output_dir    = str(frames_for_recon),
        #res           = res,
        interval_sec  = interval,
        search_window = window,
        min_sharpness = min_sharp,
        start_frame  = start_3D_recon, 
        end_frame    = end_3D_recon 

    )

    print(f"\n{len(results)} frames written to {frames_for_recon}")
else:
    print("Cell disabled")

In [ ]:
#VGGT OMEGA 
#Decide which tool to use - based on #of frames
#if  end_3D_recon - start_3D_recon < 150:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
from run_models.E_VGGT_omega import run_vggt_omega
run_vggt_omega(image_dir=str(frames_for_recon), 
               output_dir=VGGT_O_output_dir,checkpoint_path=str(project_root / "Models" / "vggt-omega" / "checkpoints"
                                            / "VGGT-Omega-1B-512" / "vggt_omega_1b_512.pt"), 
vggt_omega_dir=str(project_root / "Models" / "vggt-omega"),
image_resolution=512, conf_thres=20.0, max_points=0, show_cam=True, )

In [ ]:
#LOAD VGGT_O recon for further processing
from Thr3D.F_post_recon_processing import Reconstruction
recon = Reconstruction.load(frames_for_recon, predictions_path(), FRAME_NAME_FMT)
# CAMERA
frame_keys = sorted(recon.frame_to_row, key=recon.frame_to_row.get)
rows = [recon.frame_to_row[f] for f in frame_keys]

extrinsic = recon.preds["extrinsic"][rows].cpu().numpy()   # (N,3,4) model-to-cam
intrinsic = recon.preds["intrinsic"][rows].cpu().numpy()   # (N,3,3)

R = extrinsic[:, :3, :3]
t = extrinsic[:, :3, 3]
cam_pos = -np.einsum("nji,nj->ni", R, t)  # cam centre in model space, per frame
from Thr3D.F_post_recon_processing import positions_to_frame_dict
cam_poses_model_dict = positions_to_frame_dict(cam_pos, frames_for_recon)


In [32]:
#convert CAMERA POSES from RS model-world-space (set to CAM0) into real-world-space 
if flags["RECON_CAM_POSES"] == True and flags["VGGT_O_RECON"] ==True:
    
    #get lat lon positions
    #get transforms to convert other things too
    from Q_Metric_georeferencing import model_to_GPS_calibrated_locations
    cam_poses_realworld_dict, cam_latlons, R_cw, s_cw, t_cw, lat_0, lon_0 = model_to_GPS_calibrated_locations (csv_path_GPS, csv_path_pose)
else:
    print("Cell disabled")


Cell disabled


In [ ]:
#MULTI MERGE and PLY maker
if flags["MULTI_PART_RECON"] == True:
    
    from Thr3D.F_post_recon_processing import Reconstruction
    #Takes the lat and long to do mapping
    #Takes the cam poses dict to conform the 3d model to the same space that the cam pose transform is in
    if VGGT_O_RECON == True:
        if MULTI_PART_RECON == True:
            from B_VGGT_O_shards import discover_vggt_omega_shards
            from F_transforms_alignments import umeyama_align, recon_to_recon_transform, combine_transforms
            print("A")
            shards = discover_vggt_omega_shards(VGGT_O_output_dir)
            print("B")
            #makes list of recons.
            recons = [
                Reconstruction.load(frames_for_recon, npz_path, FRAME_NAME_FMT, frame_range=(start, end))
                for start, end, npz_path in shards
            ]
            print(f"loaded {len(recons)} recon shards: {[(s, e) for s, e, _ in shards]}")
            multi = True
            #transforms to get separate recons to align with each other - save until all transforms are made
            print("merging")
            #get tranform needed to turn first recon from modelspace into real world
            R_mw , s_mw, t_mw  = recon_to_recon_transform(recons[0],cam_poses_realworld_dict)
            merge_xforms = [(R_mw , s_mw, t_mw)]
            
            #use overlapping frames to align the recon shards
            #start with the model to world
            R = R_mw
            s = s_mw
            t = t_mw
            #for poitn clouds: we don't want the real world
            Rpc = np.eye(3)
            spc = 1
            tpc = 0
            point_cloud_xforms = [(Rpc , spc, tpc)]
        
            for i in range(len(recons)):
                #for point cloud rendering
                #rotate, scale and transform the points into real-world space and then transform to the very first camera's position
                centre = recons[i].VGGT_O_preds_to_ply_export(VGGT_O_output_dir  / "ply" , R=R, t=t, s=s, multi=multi, centre_to_cam00=t_mw)
                #load up same frame's points from the 2 recons that overlap
                #now to prepare the transform for the next shard
                if i+1 <len(recons): 
                    #EXTRINSICS of the one that's already been done
                    #rotation+translation of final frame into model space when i =0, this is cam 0
                    extr = recons[i].preds["extrinsic"][-1].cpu().numpy()      #(3,4) model->cam matrix 
                    R_cm = extr[:3, :3].T                               #cam-to-model rotation 
                    t_cm= -R_cm @ extr[:3, 3]                   #cam to model translation

                    #extrinsics of the one to be done are all 00000, so we don't need it.
                    #but we do need scale: 
                    #  fx/fy ratios they won't match - since VGGT normalises
                    intr_to   = recons[i].preds["intrinsic"][-1].cpu().numpy()   
                    intr_from = recons[i+1].preds["intrinsic"][0].cpu().numpy()  
                    fx_ratio = intr_from[0, 0] / intr_to[0, 0]
                    fy_ratio = intr_from[1, 1] / intr_to[1, 1]
                    if abs(fx_ratio - fy_ratio) / min(fx_ratio, fy_ratio) > 0.05:
                        print(f"WARNING junction {i}->{i+1}: fx/fy scale ratios disagree by >5% "
                            f"(fx={fx_ratio:.4f}, fy={fy_ratio:.4f}) - inspect this shared frame")
                    s_cm = np.sqrt(fx_ratio * fy_ratio)   #geometric mean = least-squares combiner in log-space for a multiplicative factor
                
                    #then merge into 1 set of transforms
                    R,s,t = combine_transforms(R_cm, s_cm, t_cm, R, s, t) 
                    #point cloud transformers
                    Rpc, spc, tpc = combine_transforms(R_cm, s_cm, t_cm, Rpc, spc, tpc) 
                    merge_xforms.append((R,s,t))
                    point_cloud_xforms.append((Rpc, spc, tpc)) 


                    

In [ ]:
#MULTI SUBJECT ANALYSIS
if flags["MULTI_PART_RECON"] == True:
    from Q_subject_analysis import subject_to_metric_and_gps_space
    from F_transforms_alignments import merge_pos
    #transform all the positions into real world, separately -- same 3-way
    #unpack as the singular (MULTI_PART_RECON == False) path below, just once per shard
    sub_latlons, sub_poss, sub_pos_real_dicts = {}, {}, {}
    for i in range(len(recons)):
        sub_latlons[i], sub_poss[i], sub_pos_real_dicts[i] = subject_to_metric_and_gps_space(
            recons[i],
            masks_dir=str(yolo_masks_dir),
            lat_0=lat_0, lon_0=lon_0,
            R_mw=merge_xforms[i][0], s_mw=merge_xforms[i][1], t_mw=merge_xforms[i][2],
        )

    #then combine each of the three across shards
    sub_latlon = [item for lst in sub_latlons.values() for item in lst]
    sub_pos = np.concatenate(list(sub_poss.values()))
    sub_pos_real_dict = merge_pos(*sub_pos_real_dicts.values())

In [ ]:
conf     = recon.preds["depth_conf"].cpu().numpy()
print(conf.min(), conf.max())
print(conf.shape)
print(np.percentile(conf, [5, 25, 50, 75, 95]))

In [ ]:
#Convert into model space and export into plys
#Takes the cam poses dict to conform the 3d model to the same space that the cam pose transform is in
ext = recon.preds["extrinsic"][0].clone()

if MULTI_PART_RECON == False:
        #MAKE PLYS of VGGT__O
        #turn VGGT_O output into cam0 space then scale it to metres using the output from subject_to_metric_and_gps_space
       #n.b it translates to the beginning of the camera track solve, then re-translates to the recon cam 0 space. 
       
        _ = recon.VGGT_O_preds_to_ply_export(VGGT_O_output_dir  / "ply" , 
                                             conf_threshold=1.1, 
                                             R=ext[:,0:3], 
                                             t=ext[:,3], 
                                             s=1)
        
        print(VGGT_O_output_dir / "predictions.npz")
else:
        print("Cell disabled")


In [ ]:
#processes spatial information about the recon and the cubject  - for report and generates the data needed to map the subject
#Takes the lat and long to do mapping
#Takes the cam poses dict to conform the 3d model to the same space that the cam pose transform is in
MULTI_PART_RECON = False
if flags["RECON_CAM_POSES"] == True and flags["VGGT_O_RECON"] ==True:
    if MULTI_PART_RECON == False:
        from Q_subject_analysis import subject_to_metric_and_gps_space
        from Thr3D.G_transforms_alignments import  recon_to_recon_transform
        R_mw , s_mw, t_mw  = recon_to_recon_transform(recon ,  cam_poses_realworld_dict)

        sub_latlon, sub_pos, sub_pos_real_dict = subject_to_metric_and_gps_space(
            recon,
            masks_dir        = str(yolo_masks_dir),
            lat_0=lat_0, lon_0=lon_0,
            R_mw=R_mw, s_mw =s_mw, t_mw =t_mw
            )    

        #MAKE PLYS of VGGT__O
        #turn VGGT_O output into cam0 space then scale it to metres using the output from subject_to_metric_and_gps_space
       #n.b it translates to the beginning of the camera track solve, then re-translates to the recon cam 0 space. 
        _ = recon.VGGT_O_preds_to_ply_export(VGGT_O_output_dir  / "ply" , R=R_mw, t=t_mw, s=s_mw)
        
        print(VGGT_O_output_dir / "predictions.npz")
    else:
        print("Cell disabled")
else:
     print("Cell disabled")

In [ ]:
# Cell — PLY visualisation
if flags["CUT3R_RECON"] == True and flags["RECON_3D"] == True:
        
    import importlib, sys
    sys.path.insert(0, str(project_root / "src"))

    import D_cut3r_vis; importlib.reload(D_cut3r_vis)
    from D_cut3r_vis import visualise_cut3r
    visualisation_dir = case_dir   /  VGGT_O_output_dir
    #visualisation_dir = case_dir   /  CUT3R_output_dir
    visualise_cut3r(visualisation_dir)


In [ ]:
##Graphing and analysis  
from Q_Metric_georeferencing import speed_direction, _speed_direction_single
from Q_subject_analysis import subject_reporting
fps = 30.07 #temp!
#metrics = speed_direction(cam_poses_realworld_dict, fps)
#sub_metrics = speed_direction(sub_pos_real_dict  , fps)
group_metrics = speed_direction((cam_poses_realworld_dict,sub_pos_real_dict),
                                 fps,                                 
                                 )
subject_details =   subject_reporting(sub_pos_real_dict,cam_poses_realworld_dict,lat_0, lon_0 )



In [ ]:
#graph postions 2d
from matplotlib import pyplot as plt
# sub_rw_dicts[i] == (sub_latlon_i, sub_pos_i, sub_pos_real_dict_i)
pts = np.array([sub_pos_real_dict[k] for k in sorted(sub_pos_real_dict)])
x_0 = pts[:, 0]
y_0 = pts[:, 2]

#y_0 = sub_pos_real_dict[1][:,2]
#x_0 = sub_pos_real_dict[1][:,0]
#y_1 = sub_rw_dicts[1][1][:,2]
#x_1 = sub_rw_dicts[1][1][:,0]
#y_2 = sub_rw_dicts[2][1][:,2]
#x_2 = sub_rw_dicts[2][1][:,0]
fig, ax = plt.subplots()
fig.patch.set_facecolor("black")
ax.set_facecolor("black")

ax.scatter(x_0, y_0, c=np.arange(len(x_0)), cmap="Reds")
#ax.scatter(x_1, y_1, c=np.arange(len(x_1)), cmap="Greens")
#ax.scatter(x_2, y_2, c=np.arange(len(x_2)), cmap="Blues")

ax.tick_params(colors="white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.title.set_color("white")
ax.set_xlabel("east/west (m)")
ax.set_ylabel("north/south (m)")
ax.set_title("Subject position by shard (color = time)")


In [ ]:
#make animated map -- always writes an image sequence to assets/map_frames
# (the standard hand-off to the editor); gif/mp4 below are optional extra
# exports of the same frames. Duration is read from the Producer's MAP beat
# inside R_map_animator.main() itself, not passed in here.
if flags["GOOGLE_MAP"] == True:
    import Rendering.R_map_animator

    R_map_animator.main(
        traces=[
            {"coords": cam_latlons, "trail_color": "#3498db"},#cameras
            {"coords": sub_latlon,    "trail_color": "#e74c3c"},#subject
        ],
        api_key     = api_key,
        gif = True,
        mp4 = None
        

    )
else:
    print("Cell disabled")

In [ ]:
#CAMERA TRANSFORMS - single recon at the moment
#1m up
ext = recon.preds["extrinsic"][9].clone()
ext[1, 3] += 1/16   # lift camera 1m (1/16 scale)

#10m up BEV
ext9 = recon.preds["extrinsic"][9]
Rx = torch.tensor([[1., 0., 0.],
                   [0., 0., -1.],
                   [0., 1., 0.]], dtype=ext9.dtype, device=ext9.device)

ext = Rx @ ext9        # pitch 90° down, camera stays where cam 9 was
ext[2, 3] += 20/16     # lift 10m (up is -z now that the view axis points down)
ext[1, 3] += 5/16      # forward 5m along cam 9's original heading

In [ ]:
if flags["PROJECTION_MAP"] == True:
    #2D projection mapping of subject to a selected view
    from Thr3D.F_post_recon_processing import Reconstruction
    from P_projection_mapping import projection_mapping_sequence
    from Two2D.B_video_processing import available_frames
    
    #Use this to project every frame in the recon
    extra_indices = available_frames(frames_for_recon, FRAME_NAME_FMT)
    print(extra_indices)
    
    
    if MULTI_PART_RECON == True:
      recon = recons
      point_cloud_xforms = point_cloud_xforms
    else: 
      recon = recon
      point_cloud_xforms = None
    subject_positions = projection_mapping_sequence(
        
        recon         = recon,
        main_idx      = 3908,
        extra_indices = extra_indices,
        extra_indices = range (3854,4397),
        #other syntax  [400,4027, 4052, 4065],
        #extra_indices = [extra_indices], - for composite (double [[]])
        masks_dir     = str(yolo_masks_dir),
        point_cloud_xforms = point_cloud_xforms

        new_view = ext ,
        
        confidence_threshold = 2
       
       )
else:
    print("Cell disabled")

In [ ]:
#render projection of 3D points clouds
from Rendering.U_rendering import basic_point_cloud_render
#double way to get vggt and cut3r
if flags["VGGT_O_RECON"] ==True:
    visualisation_dir = case_dir   /  VGGT_O_output_dir 
    model = "VGGT_O"
   

    ply_dir = visualisation_dir / "ply"
    print(ply_dir)
    basic_point_cloud_render(
        ply_dir, model    )
    # report metric name = render name, one row per output asset (path relative to
    


In [ ]:

#cutr point cloud render
if flags["CUT3R_RECON"] ==True:  
    visualisation_dir = case_dir   /  CUT3R_output_dir 
    model = "CUT3R"
    ply_dir = visualisation_dir / "ply"
    basic_point_cloud_render(
        ply_dir, model     )
    # report metric name = render name, one row per output asset (path relative to
   



In [ ]:
#4D viewer
if flags["VGGT_O_RECON"] ==True:
     ply_dir = VGGT_O_output_dir  /"ply"  
from Rendering.U_rendering import view_ply_sequence_with_scenepic
FourD =  view_ply_sequence_with_scenepic(
    ply_dir , 
    point_size=0.01, 
    pattern="*.ply")

In [ ]:
#get times of day into the report
from C_CSV_report import attach_times_of_day
report = attach_times_of_day(video_path, 30.07)


In [ ]:
if flags["MAKE_MOVIE"] ==True:
  #MAKING A MOVIE
  '''This controls M_scoring. It's independent of scene recon, so could be placed anywhere'''
  '''bring in M_scoring_controls'''
  from Two2D.W_video_editor import video_overlay_edit
  movie_start = movie_start
  moveie_end = movie_end
  movie =video_overlay_edit(video_path, 
                            movie_start, 
                            moveie_end, 
                            yolo_masks_dir, 
                            3777, 4557,
                            handles=2, 
                          color=(0, 200, 0), 
                          alpha=0.4) 

In [ ]:
#find the cut points
if flags["MAKE_MOVIE_CUTS"] == True:
    import Two2D.W_video_editor
    W_video_editor.find_all_cut_points(video_path, analysis_2d_for_decisions)

In [ ]:
#run producer again
sys.path.insert(0, str(project_root / "src" / "claude_agent"))
from producer_agent_execution import run_producer_agent, build_producer_prompt_revision

prompt = build_producer_prompt_revision(Question, analysis_2d_for_decisions, producer_flags)
paper_edit_json_path = run_producer_agent(prompt)


In [ ]:
#make mask overlays and return image sequence
from Two2D.W_video_editor import mask_compositor
tracker = "SAM3"
movie_start = 225*30 
movie_end = 237*30
mask_compositor(video_path, 
                            start = movie_start, 
                            end = movie_end, 
                            tracker= tracker, 
                            mask_start = movie_start,
                            mask_end = movie_end,
                            handles=2, 
                          color=(0, 200, 0), 
                          alpha=0.4) 


In [ ]:
#Make the edited multimedia movie
if flags["MAKE_MOVIE_CUTS"] == True:
    import Two2D.W_video_editor
 
    W_video_editor.assemble_paper_edit(video_path)
else:
    print("Cell disabled")

In [ ]:
from V_text_summary import gemini_final_report, gemini_cache_video
from A_Config import assets_dir
video_path = assets_dir() / f"{asset_name}_overlay_video.mp4"
cache =  gemini_cache_video(video_path, assets_dir(), ttl="6000s")
gemini_final_report()

In [ ]:
# Cell — Populate PowerPoint from CSV
import sys
import importlib
sys.path.insert(0, str(project_root / "src"))

import W_populate_pptx
importlib.reload(W_populate_pptx)
from W_populate_pptx import populate



report_template = project_root / "src" / "Powerpoint" / f"{ppt_template}.pptx"
ppt_assets   = case_dir / "090_Assets"
ppt_csv      = ppt_assets / f"{case_name}.csv"
ppt_output   = case_dir / "100_Powerpoint" / f"{case_name}.pptx"


populate(str(report_template))
